In [1]:
# CELL 0 – dùng chung cho NB01 → NB07
import os, json
from pathlib import Path

# notebooks/ nằm dưới thư mục gốc 1 cấp
PROJECT_ROOT = Path(os.getcwd()).parent
CONFIG_PATH = PROJECT_ROOT / "src" / "config.json"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy {CONFIG_PATH}. Hãy chạy 00_prep_features.ipynb trước.")

with open(CONFIG_PATH, "r", encoding="utf-8") as f:
    cfg = json.load(f)

# Dùng Path để ghép đường dẫn cho tiện
RAW      = Path(cfg["RAW"])       # chỉ dùng ở NB01
FEATURES = Path(cfg["FEATURES"])
RESULTS  = Path(cfg["RESULTS"])
FIGURES  = Path(cfg["FIGURES"])

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW     :", RAW)
print("FEATURES:", FEATURES)
print("RESULTS :", RESULTS)
print("FIGURES :", FIGURES)


PROJECT_ROOT: D:\STAT3013.Q12_Group01
RAW     : D:\STAT3013.Q12_Group01\data\raw
FEATURES: D:\STAT3013.Q12_Group01\features
RESULTS : D:\STAT3013.Q12_Group01\results
FIGURES : D:\STAT3013.Q12_Group01\figures


In [2]:
import pandas as pd

tx = pd.read_parquet(FEATURES / "tx_clean.parquet")
tx.head()


,household_key,basket_id,day,product_id,quantity,sales_value,store_id,retail_disc,trans_time,week_no,coupon_disc,coupon_match_disc,unit_price,department,commodity_desc,sub_commodity_desc,display,mailer
0,2375,26984851472,1,1004906,1,1.39,364,-0.60,1631,1,0.0,0.0,1.39,PRODUCE,POTATOES,POTATOES RUSSET (BULK&BAG),0,0
1,2375,26984851472,1,1033142,1,0.82,364,0.00,1631,1,0.0,0.0,0.82,PRODUCE,ONIONS,ONIONS SWEET (BULK&BAG),0,0
2,2375,26984851472,1,1036325,1,0.99,364,-0.30,1631,1,0.0,0.0,0.99,PRODUCE,VEGETABLES - ALL OTHERS,CELERY,0,0
3,2375,26984851472,1,1082185,1,1.21,364,0.00,1631,1,0.0,0.0,1.21,PRODUCE,TROPICAL FRUIT,BANANAS,0,0
4,2375,26984851472,1,8160430,1,1.50,364,-0.39,1631,1,0.0,0.0,1.50,PRODUCE,ORGANICS FRUIT & VEGETABLES,ORGANIC CARROTS,0,0


In [3]:
for col in ["display", "mailer", "price_reduction"]:
    if col not in tx.columns:
        tx[col] = 0.0
    else:
        tx[col] = tx[col].fillna(0.0)

# --- Gom nhóm dữ liệu ---
orders = (
    tx.groupby(["household_key","basket_id"], as_index=False)
      .agg(
          day=("day","min"),
          week_no=("week_no","min"),
          store_id=("store_id","min"),
          basket_value=("sales_value","sum"),
          basket_qty=("quantity","sum"),
          retail_disc=("retail_disc","sum"),
          coupon_disc=("coupon_disc","sum"),
          coupon_match_disc=("coupon_match_disc","sum"),
          n_items=("product_id","nunique"),
          n_dept=("department","nunique"),
          # Bây giờ ta có thể lấy mean trực tiếp vì đã fill 0 ở trên
          promo_display=("display","mean"),
          promo_mailer=("mailer","mean"),
          promo_price_red=("price_reduction","mean"),
      )
)

orders = orders.sort_values(["household_key","day"])
orders["avg_price"] = orders["basket_value"] / orders["basket_qty"].replace(0,1)
orders["treatment"] = (
    (orders.coupon_disc != 0) | (orders.coupon_match_disc != 0)
).astype(int)

print("Orders shape:", orders.shape)
orders.head()

Orders shape: (113574, 17)


,household_key,basket_id,day,week_no,store_id,basket_value,basket_qty,retail_disc,coupon_disc,coupon_match_disc,n_items,n_dept,promo_display,promo_mailer,promo_price_red,avg_price,treatment
0,1,27601281299,51,8,436,78.66,34,-16.54,-1.0,0.0,30,6,0.0,0.0,0.0,2.313529,1
1,1,27774192959,67,10,436,41.10,14,-8.59,0.0,0.0,11,3,0.0,0.0,0.0,2.935714,0
2,1,28024266849,88,13,436,26.90,13,-6.72,0.0,0.0,12,4,0.0,0.0,0.0,2.069231,0
3,1,28106322445,94,14,436,63.43,32,-11.08,-0.5,-0.5,22,3,0.0,0.0,0.0,1.982187,1
4,1,28235481967,101,15,436,53.45,20,-16.42,0.0,0.0,17,5,0.0,0.0,0.0,2.672500,0


In [4]:
orders["next_day"] = orders.groupby("household_key")["day"].shift(-1)
orders["gap_days"] = orders["next_day"] - orders["day"]
orders["repeat30d"] = (orders["gap_days"] <= 30).fillna(False).astype(int)


In [5]:
orders = orders.sort_values(["household_key", "day"]).copy()

orders["prev_day"] = orders.groupby("household_key")["day"].shift(1)
orders["recency"] = (orders["day"] - orders["prev_day"]).fillna(999)

orders["frequency"] = orders.groupby("household_key").cumcount()

orders["monetary_mean_prev"] = (
    orders.groupby("household_key")["basket_value"]
          .transform(lambda s: s.shift(1).expanding().mean())
)

orders["monetary_mean_prev"] = orders["monetary_mean_prev"].fillna(orders["basket_value"])

first_day = orders.groupby("household_key")["day"].transform("min")
orders["tenure"] = orders["day"] - first_day

orders["dow"] = (orders["day"] % 7).astype(int)
orders["weekofyear"] = (orders["week_no"] % 52).astype(int)


In [6]:
total_disc = (
    orders["retail_disc"].abs()
    + orders["coupon_disc"].abs()
    + orders["coupon_match_disc"].abs()
)

gross_value = orders["basket_value"] + total_disc   

orders["discount_rate"] = (total_disc / gross_value.replace(0,1)).clip(0,1)


In [7]:
FEATURES.mkdir(exist_ok=True)
out_path = FEATURES / "repeat30d.parquet"
orders.to_parquet(out_path, index=False)
print("Saved repeat30d.parquet to:", out_path)


Saved repeat30d.parquet to: D:\STAT3013.Q12_Group01\features\repeat30d.parquet
